In [1]:
import sys
print(sys.version)
print(sys.executable)

import cv2
import mediapipe as mp
import pandas as pd

print("cv2:", cv2.__version__)
print("mediapipe:", mp.__version__)
print("has solutions:", hasattr(mp, "solutions"))

3.11.15 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:12:53) [MSC v.1942 64 bit (AMD64)]
c:\Users\Mahmo\anaconda3\envs\hand_gesture\python.exe
cv2: 4.11.0
mediapipe: 0.10.21
has solutions: True


In [5]:
import cv2

try:
    cap.release()
except:
    pass

cv2.destroyAllWindows()
print("Cleaned.")

Cleaned.


In [6]:
import cv2
import mediapipe as mp
import csv
import os
from collections import deque
from datetime import datetime
import pandas as pd

GESTURE_LABEL = "left_click"
OUTPUT_FILE = "gestures_seq.csv"
WINDOW = 15
WINDOW_NAME = "Recorder"

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
)
mp_draw = mp.solutions.drawing_utils

cap = cv2.VideoCapture(0, cv2.CAP_DSHOW)

if not cap.isOpened():
    print("Camera not opened. Try changing camera index from 0 to 1.")
    raise SystemExit

if not os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "w", newline="") as f:
        header = (
            ["sample_id", "label", "source_video"]
            + [
                f"t{t}_{i}_{a}"
                for t in range(WINDOW)
                for i in range(21)
                for a in ["x", "y", "z"]
            ]
        )
        csv.writer(f).writerow(header)

def normalize(landmarks):
    wrist = landmarks[0]

    pts = [
        (lm.x - wrist.x, lm.y - wrist.y, lm.z - wrist.z)
        for lm in landmarks
    ]

    scale = max(
        (px**2 + py**2 + pz**2) ** 0.5
        for px, py, pz in pts
    ) or 1.0

    return [v / scale for p in pts for v in p]

buffer = deque(maxlen=WINDOW)
count = 0
session_tag = datetime.now().strftime("%Y%m%d_%H%M%S")

print(f"Recording gesture: {GESTURE_LABEL}")
print("Press SPACE to save.")
print("Press Q, ESC, or close the window to quit.")

cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)

try:
    while True:
        ret, frame = cap.read()

        if not ret:
            print("Failed to read from camera.")
            break

        frame = cv2.flip(frame, 1)

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        if result.multi_hand_landmarks:
            lms = result.multi_hand_landmarks[0]

            mp_draw.draw_landmarks(
                frame,
                lms,
                mp_hands.HAND_CONNECTIONS
            )

            buffer.append(normalize(lms.landmark))

        cv2.putText(
            frame,
            f"Gesture: {GESTURE_LABEL} | Saved: {count} | Buffer: {len(buffer)}/{WINDOW}",
            (10, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.7,
            (0, 255, 0),
            2
        )

        cv2.imshow(WINDOW_NAME, frame)

        key = cv2.waitKey(30) & 0xFF

        if key == ord(" ") and len(buffer) == WINDOW:
            count += 1
            sample_id = f"{GESTURE_LABEL}_{session_tag}_{count}"

            row = (
                [sample_id, GESTURE_LABEL, "webcam_live"]
                + [v for frame_vec in buffer for v in frame_vec]
            )

            with open(OUTPUT_FILE, "a", newline="") as f:
                csv.writer(f).writerow(row)

            buffer.clear()
            print(f"Saved sequence #{count}")

        if key == ord("q") or key == 27:
            print("Stopping recorder...")
            break

        if cv2.getWindowProperty(WINDOW_NAME, cv2.WND_PROP_VISIBLE) < 1:
            print("Window closed.")
            break

finally:
    cap.release()
    hands.close()
    cv2.destroyAllWindows()
    cv2.waitKey(1)
    print("Camera closed.")

if os.path.exists(OUTPUT_FILE):
    df = pd.read_csv(OUTPUT_FILE)

    print("\nClass balance so far:")
    print(df["label"].value_counts())

    print("\nSaved file:", OUTPUT_FILE)

Recording gesture: left_click
Press SPACE to save.
Press Q, ESC, or close the window to quit.
Stopping recorder...
Camera closed.

Class balance so far:
label
left_click    1
Name: count, dtype: int64

Saved file: gestures_seq.csv
